# Quant Factor Mining Agent developer example

In this notebook, we showcase how the NeMo Agent toolkit can be used to create an end-to-end factor mining workflow for quantitative finance. This workflow demonstrates how to leverage LLMs to design a automated system to generate, code, and evaluate alpha factors.

## What You'll Learn

By the end of this notebook, you will understand how to:
- Set up a multi-step sequential workflow using NAT
- Implement custom functions for factor generation, code generation, and evaluation
- Use LLMs for creative factor design and code synthesis
- Evaluate factor performance using rank IC metrics
- Run complete end-to-end factor mining workflows with optimization loops

## Workflow Overview

This factor mining workflow consists of three agents orchestrated by the NeMo Agent Toolkit:

<div style="background-color: #e8f5e9; border-left: 6px solid #4caf50; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🧩 Workflow Components</h4>
<ul>
<li><strong>Factor Agent</strong> — Uses an LLM to generate creative factor expressions based on price-volume data and predefined operators</li>
<li><strong>Code Agent</strong> — Converts factor expressions into executable Python code, leveraging the <code>Get_calculator</code> tool for operator implementations</li>
<li><strong>Eval Agent</strong> — Evaluates the predictive power of generated factors using Rank IC (backtesting) and provides optimization suggestions for iterative improvement</li>
</ul>
All three agents are powered by the <strong>nvidia/nvidia/llama-3.3-nemotron-super-49b-v1.5</strong> NIM model.
</div>

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⚠️ Note</h4>
This example demonstrates the capabilities of NeMo Agent Toolkit for automating quantitative research workflows. The generated factors should be thoroughly validated before use in production.
</div>

## Table of Contents

- [0) Setup](#setup)
  - [0.1) Prerequisites](#prereqs)
  - [0.2) API Keys](#api-keys)
  - [0.3) Installing Dependencies](#installing-deps)
  - [0.4) Download Data](#download-data)
- [1) Understanding Factor Mining](#understanding-workflow)
  - [1.1) What is Factor Mining?](#what-is-factor-mining)
  - [1.2) Workflow Architecture](#workflow-architecture)
- [2) Source Code Components](#source-code)
  - [2.1) Factor Agent](#factor-agent)
  - [2.2) Code Agent](#code-agent)
  - [2.3) Eval Agent](#eval-agent) *(includes Rank IC utilities)*
  - [2.4) Factor Mining Optimization Workflow](#optimization-workflow)
  - [2.5) Register Module](#register-module)
- [3) Configuration](#configuration)
  - [3.1) Workflow Configuration](#workflow-config)
  - [3.2) Operator Templates](#operator-templates)
- [4) Running the Workflow](#running-workflow)
  - [4.1) Via Command Line](#run-cli)
  - [4.2) Programmatically in Python](#run-python)
- [5) Interpreting Results](#interpreting-results)
  - [5.1) Understanding Evaluation Metrics](#evaluation-metrics)
  - [5.2) Analyzing Generated Factors](#analyzing-factors)
- [6) Next Steps](#next-steps)

<a id="setup"></a>
# 0) Setup

<a id="prereqs"></a>
## 0.1) Prerequisites

<div style="background-color: #e8f4f8; border-left: 6px solid #0288d1; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📋 Requirements</h4>
<ul>
<li><strong>Platform:</strong> Linux, macOS, or Windows</li>
<li><strong>Python:</strong> version 3.11, 3.12, or 3.13</li>
<li><strong>Package manager:</strong> pip or uv</li>
</ul>
</div>

<a id="api-keys"></a>
## 0.2) API Keys

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🔑 NVIDIA API Key Required</h4>
For this notebook, you will need an NVIDIA API key. Get yours from <a href="https://build.nvidia.com/settings/api-keys">build.nvidia.com</a>.
</div>

In [1]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

<a id="installing-deps"></a>
## 0.3) Installing Dependencies

Install the factor mining workflow package:

In [2]:
# Install the factor mining workflow package
# Run this from the notebooks directory (one level up to project root)
%pip install -e ..

Obtaining file:///Users/phuo/Desktop/quant-factor-mining-agent
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for factor_mining_workflow (pyproject.toml) ... done
  Created wheel for factor_mining_workflow: filename=factor_mining_workflow-0.0.0-0.editable-py3-none-any.whl size=6315 sha256=a0c2666c7936f5989ad296e33c7655cc81fc12e0ce4816bf4e889d8c864cba99
  Stored in directory: /private/var/folders/nt/4r3np6cn6t19q2njfd1xm20h0000gp/T/pip-ephem-wheel-cache-64jxfty2/wheels/fb/0b/e9/7dc1fc6cb554490678142c021ea4596f95b315c0647c3fbc0e
Successfully built factor_mining_workflow
  Attempting uninstall: factor_mining_workflow
    Found existing installation: factor_mining_workflow 0.0.0
    Uninstalling factor_mining_workflow-0.0.0:
      Successfully uninstalled factor_mining_workflow-0.0.0

[notice] A new release of p

<a id="download-data"></a>
## 0.4) Download Data

<div style="background-color: #e8f4f8; border-left: 6px solid #0288d1; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📥 S&P 500 Price-Volume Data</h4>
The workflow requires price-volume data (Open, Close, High, Low, Volume) stored as CSV files in <code>src/factor_mining_workflow/data/sp500/</code>. Use the script below to download fresh data from Yahoo Finance via <code>yfinance</code>.
</div>

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⚠️ Disclaimer</h4>
Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.
</div>

In [ ]:
from factor_mining_workflow.download_data import download_sp500_data

data = download_sp500_data(start="2012-01-01", end="2025-12-31")

for field, df in data.items():
    print(f"{field}: {df.shape[0]} days x {df.shape[1]} stocks")

<a id="understanding-workflow"></a>
# 1) Understanding Factor Mining

<a id="what-is-factor-mining"></a>
## 1.1) What is Factor Mining?

Factor mining is the process of discovering quantitative signals (factors) that have predictive power for future stock returns. In quantitative finance, factors are numerical characteristics of stocks that can be used to:

- **Predict returns**: Factors with high Information Coefficient (IC) can help forecast which stocks will outperform
- **Construct portfolios**: Long stocks with favorable factor values, short those with unfavorable values
- **Manage risk**: Understand exposures to different sources of returns

Traditional factor research is labor-intensive, requiring researchers to:
1. Formulate hypotheses about what might predict returns
2. Implement the factor calculation in code
3. Backtest the factor's performance
4. Iterate based on results

This workflow automates this process using LLMs!

<a id="workflow-architecture"></a>
## 1.2) Workflow Architecture

![Workflow Architecture](images/workflow-architecture.png)

<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 Closed-Loop Optimization</h4>
The workflow is orchestrated by the <strong>NeMo Agent Toolkit</strong> and uses a closed-loop optimization approach:
<ol>
<li><strong>Factor Agent</strong> receives price-volume data and operators, then generates factor expressions using the LLM</li>
<li><strong>Code Agent</strong> converts factor expressions into executable Python code, using the <code>Get_calculator</code> tool to access operator implementations</li>
<li><strong>Eval Agent</strong> evaluates the factor's predictive power via backtesting (Rank IC)
<ul>
<li>If IC meets threshold → Accept and produce <strong>backtesting results</strong></li>
<li>If IC is poor → Generate <strong>optimization suggestions</strong> and feed them back to the Factor Agent for a new iteration</li>
</ul>
</li>
</ol>
</div>

<a id="source-code"></a>
# 2) Source Code Components

Below we load and display the source code for each component of the factor mining workflow. Use `%load` to load the Python source files.

<a id="factor-agent"></a>
## 2.1) Factor Agent

<div style="background-color: #e8f5e9; border-left: 6px solid #4caf50; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📊 Factor Agent</h4>
Uses the <strong>nvidia/nvidia/llama-3.3-nemotron-super-49b-v1.5</strong> NIM to generate quantitative factor expressions based on available price-volume data (Open, Close, High, Low, Volume) and predefined operators.
<ul>
<li><strong>Input:</strong> Factor type request (e.g., "momentum factors"), price-volume data fields, operator definitions</li>
<li><strong>Output:</strong> Factor expressions in JSON format with names, formulas, and economic intuition</li>
</ul>
</div>

In [21]:
# %load ../src/factor_mining_workflow/factor_generator.py

<a id="code-agent"></a>
## 2.2) Code Agent

<div style="background-color: #e3f2fd; border-left: 6px solid #1976d2; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💻 Code Agent</h4>
Converts factor expressions into executable Python code. Uses the LLM to synthesize the factor function and leverages the <code>Get_calculator</code> tool to automatically include the required operator implementations from <code>calculator.json</code>.
<ul>
<li><strong>Input:</strong> Factor expressions (JSON) from the Factor Agent</li>
<li><strong>Output:</strong> Executable Python code implementing the factor calculations</li>
<li><strong>Tool:</strong> <code>Get_calculator</code> — retrieves operator implementations</li>
</ul>
</div>


In [ ]:
# %load ../src/factor_mining_workflow/factor_code_generator.py


<a id="eval-agent"></a>
## 2.3) Eval Agent

<div style="background-color: #fff3e0; border-left: 6px solid #ff9800; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📈 Eval Agent</h4>
Performs backtesting by computing the Spearman correlation (Rank IC) between factor values and forward stock returns to measure predictive power.
<ul>
<li><strong>Input:</strong> Executable codes from the Code Agent</li>
<li><strong>Output:</strong> Backtesting results (IC metrics) and optimization suggestions</li>
<li><strong>Accept criteria:</strong> |IC| ≥ threshold and p-value ≤ significance level</li>
<li><strong>Feedback loop:</strong> When factors fall below threshold, generates optimization suggestions for the Factor Agent</li>
</ul>
</div>

In [ ]:
# %load ../src/factor_mining_workflow/factor_evaluator.py

<a id="optimization-workflow"></a>
## 2.4) Factor Mining Optimization Workflow

<div style="background-color: #f9f9f9; border: 2px solid #ddd; padding: 20px; border-radius: 8px; margin: 20px 0;">
<h4 style="margin-top: 0;">🔄 NeMo Agent Toolkit Orchestration</h4>
The Factor Mining Optimization Workflow uses the NeMo Agent Toolkit to orchestrate the entire closed-loop pipeline. It composes the Factor Agent, Code Agent, and Eval Agent — iteratively generating factor expressions, converting them to executable codes, backtesting their performance, and feeding optimization suggestions back until the IC threshold is met.
<pre style="background-color: #f5f5f5; padding: 10px; border-radius: 3px; margin: 10px 0;">
Factor Agent → factor expressions → Code Agent → executable codes → Eval Agent
      ↑                                                                    |
      └──────────── optimization suggestions ←─────────────────────────────┘
</pre>
</div>

In [ ]:
# %load ../src/factor_mining_workflow/factor_mining_optimization_workflow.py

<a id="register-module"></a>
## 2.5) Register Module

The register module imports all components to make them available to the NeMo Agent Toolkit.

In [ ]:
# %load ../src/factor_mining_workflow/register.py

<a id="configuration"></a>
# 3) Configuration

<a id="workflow-config"></a>
## 3.1) Workflow Configuration

<div style="background-color: #e8f4f8; border-left: 6px solid #0288d1; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⚙️ Workflow Configuration</h4>
The workflow configuration defines how the factor optimization agent operates. It specifies:
<ul>
<li><strong>LLM assignments</strong> — Which model powers each agent (Factor, Code, Eval)</li>
<li><strong>Temperature settings</strong> — Higher for creative factor generation, lower for precise code generation</li>
<li><strong>Acceptance criteria</strong> — IC threshold and p-value thresholds for factor quality</li>
<li><strong>Iteration limits</strong> — Maximum optimization attempts before accepting best result</li>
</ul>
</div>

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⚠️ LLM Base URL</h4>
The <code>base_url</code> for the LLMs depends on your API key:
<ul>
<li><code>https://integrate.api.nvidia.com/v1/</code> — for keys from <a href="https://build.nvidia.com">build.nvidia.com</a></li>
<li><code>https://inference-api.nvidia.com/v1/</code> — for NVIDIA internal or enterprise API keys</li>
</ul>
Update the <code>base_url</code> field in <code>configs/config-optimization.yml</code> to match your API key type.
</div>

In [22]:
# Display the configuration file
with open("../configs/config-optimization.yml", "r") as f:
    print(f.read())

# Factor Mining Workflow
# Iteratively generates and optimizes quantitative factors:
# 1. Generate factor descriptions using LLM
# 2. Convert to executable Python code
# 3. Evaluate rank IC against forward returns
# 4. Accept if IC meets threshold, otherwise retry with feedback

# Global settings with Phoenix tracing
# Run `phoenix serve` in a separate terminal before running the workflow
general:
  telemetry:
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:6006/v1/traces
        project: factor-mining-workflow

llms:
  # Factor Generator: Creative model for generating factor ideas
  factor_generator:
    _type: nim
    model_name: nvidia/nvidia/llama-3.3-nemotron-super-49b-v1.5
    base_url: "https://inference-api.nvidia.com/v1/"
    temperature: 0.8          # Higher temperature for creativity
    max_tokens: 4096

  # Code Generator: Precise model for generating executable code
  code_generator:
    _type: nim
    model_name: nvidia/nvidia/llama

<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 Configuration Parameters Explained</h4>
<table style="width: 100%; border-collapse: collapse;">
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>factor_generator_llm</code></td><td style="padding: 8px;">LLM for generating factor expressions (higher temperature for creativity)</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>code_generator_llm</code></td><td style="padding: 8px;">LLM for converting expressions to executable code (lower temperature for precision)</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>optimization_advisor_llm</code></td><td style="padding: 8px;">LLM for generating optimization feedback (balanced temperature)</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>ic_threshold</code></td><td style="padding: 8px;">Minimum absolute IC value to accept a factor (e.g., 0.02 = 2%)</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>p_value_threshold</code></td><td style="padding: 8px;">Maximum p-value for statistical significance (e.g., 0.05 = 5%)</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>max_iterations</code></td><td style="padding: 8px;">Maximum number of optimization iterations before accepting best result</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>num_factors</code></td><td style="padding: 8px;">Number of factors to generate per iteration</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><code>forward_periods</code></td><td style="padding: 8px;">Number of days for forward return calculation (e.g., 5 = weekly)</td></tr>
<tr><td style="padding: 8px;"><code>save_results</code></td><td style="padding: 8px;">Whether to save successful factors to disk</td></tr>
</table>
</div>

<a id="operator-templates"></a>
## 3.2) Operator Templates

<div style="background-color: #e8f5e9; border-left: 6px solid #4caf50; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🧮 Operator Templates</h4>
The workflow uses predefined operators from <code>calculator.json</code>. These operators are the building blocks that the Factor Agent uses to compose factor expressions. The <code>Get_calculator</code> tool provides these implementations to the Code Agent.
</div>

In [3]:
import json
from pathlib import Path

# Load and display sample operators
template_path = Path("../src/factor_mining_workflow/template/calculator.json")
with open(template_path, "r") as f:
    operators = json.load(f)

print(f"Total operators available: {len(operators)}\n")
print("Sample operators:")
print("-" * 60)
for op in operators[:10]:
    print(f"\n{op['name']}:")
    print(f"  Signature: {op['signature']}")
    print(f"  Meaning: {op['meanings'][:80]}...")

Total operators available: 66

Sample operators:
------------------------------------------------------------

Add:
  Signature: Add(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame
  Meaning: Add two DataFrames element-wise and return the result....

Sub:
  Signature: Sub(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame
  Meaning: Subtract the second DataFrame from the first DataFrame element-wise and return t...

Mul:
  Signature: Mul(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame
  Meaning: Multiply two DataFrames element-wise and return the result....

Div:
  Signature: Div(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame
  Meaning: Divide the first DataFrame by the second DataFrame element-wise and return the r...

Rank_Add:
  Signature: Rank_Add(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame
  Meaning: Return the sum of the quantiles of the elements in DataFrame x and the correspon...

Rank_Sub:
  Signature: Rank_Sub(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame
  Me

<a id="running-workflow"></a>
# 4) Running the Workflow

<div style="background-color: #e8f5e9; border-left: 6px solid #4caf50; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🚀 Running the Workflow</h4>
Now we run the workflow and see the generated factors. The workflow will:
<ol>
<li>Start the NeMo Agent Toolkit orchestration</li>
<li>Load price-volume data (Open, Close, High, Low, Volume)</li>
<li>Run the Factor Agent → Code Agent → Eval Agent pipeline</li>
<li>Iterate until IC threshold is met or max iterations reached</li>
<li>Save accepted factors to disk</li>
</ol>
</div>

Phoenix is an open-source observability platform that captures and visualizes traces from your agent. It runs as a local web server that collects trace data and provides an interactive UI for analysis.
<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🚀 Production vs. Notebook Setup</h4>
<strong>In production,</strong> you'd simply run in a terminal:
<pre style="background-color: #f5f5f5; padding: 10px; border-radius: 3px; margin: 10px 0;">
phoenix serve
</pre>
<strong>In Jupyter,</strong> you'll use <code>subprocess</code> to start Phoenix in the background.
</div>

In [16]:
import subprocess
import time
import sys

# Start Phoenix 
phoenix_process = subprocess.Popen(
    ["phoenix", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Read initial output
print("Starting Phoenix...")
start_time = time.time()
while time.time() - start_time < 3:
    line = phoenix_process.stdout.readline()
    if line:
        print(line.strip())

print("\n✅ Phoenix is running silently in the background!")

Starting Phoenix...
🏃‍♀️‍➡️ Running migrations on the database.

✅ Phoenix is running silently in the background!


<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 What Phoenix Does</h4>
<ul>
<li><strong>Captures traces</strong> - Every agent decision, tool call, and LLM interaction</li>
<li><strong>Stores telemetry</strong> - Timing data, token usage, success/failure status</li>
<li><strong>Provides visualization</strong> - Interactive UI to explore traces and find patterns</li>
<li><strong>Runs locally</strong> - Your data never leaves your machine (default port: 6006)</li>
</ul>
</div>

## Access Phoenix UI
The following codeblock is just some quick html to get the URL for your Phoenix server: 

In [17]:
from IPython.display import HTML, display

html = """
<div style="padding: 15px; background-color: #e8f4f8; border-radius: 8px; margin: 10px 0;">
    <h4 style="margin: 0 0 10px 0;">🔍 Phoenix Observability Dashboard</h4>
    <p>Click the button to open Phoenix (it will auto-detect the correct URL):</p>
    <button onclick="
        var url = window.location.origin.replace(/p\\d+/, 'p6006');
        window.open(url, '_blank');
    " style="padding: 10px 20px; background-color: #0066cc; color: white; border: none; border-radius: 4px; cursor: pointer; font-size: 14px;">
        🚀 Open Phoenix UI
    </button>
    <p style="margin-top: 10px; font-size: 12px; color: #666;">
        If the link doesn't work, manually replace the port in your browser URL from p8888 to p6006
    </p>
</div>
"""

display(HTML(html))

## Phoenix Configuration
To enable Phoenix tracing, you add a `telemetry section` to your NAT config. This tells NAT where to send trace data.
<div style="background-color: #f9f9f9; border: 2px solid #ddd; padding: 20px; border-radius: 8px; margin: 20px 0;">
<h4 style="text-align: center;">Phoenix Configuration Structure</h4>
<pre style="background-color: #f5f5f5; padding: 15px; border-radius: 5px; margin: 10px 0;">
general:                                              # Global workflow settings
  telemetry:                                          # Monitoring and metrics
    tracing:                                          # Distributed tracing config
      phoenix:                                        # Phoenix-specific settings
        _type: phoenix                                # Use Phoenix provider
        endpoint: http://localhost:6006/v1/traces     # Where to send data
        project: climate_analyzer_baseline            # Project name in UI
</pre>
</div>
<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 Key Configuration Fields</h4>
<ul>
<li><strong>endpoint</strong> - Phoenix trace collector URL (must match where Phoenix is running)</li>
<li><strong>project</strong> - Groups traces together in the UI. Use different names to compare versions (e.g., "baseline" vs. "optimized")</li>
<li><strong>_type: phoenix</strong> - NAT supports multiple tracing backends; this specifies Phoenix</li>
</ul>
</div>
Run the code below once- the updated YAML file will display below after the telemetry section has been added. 

In [19]:
# Run the factor optimization workflow via CLI
!cd .. && nat run --config_file configs/config-optimization.yml --input "momentum factors"

2026-03-03 13:35:12 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'configs/config-optimization.yml'
2026-03-03 13:35:12 - INFO     - phoenix.config:2716 - 📋 Ensuring phoenix working directory: /Users/phuo/.phoenix
2026-03-03 13:35:12 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.schemas
2026-03-03 13:35:12 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.tables
2026-03-03 13:35:12 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.types
2026-03-03 13:35:12 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.constraints
2026-03-03 13:35:12 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.defaults
2026-03-03 13:35:12 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.comments
2026-03-03 13:35:13 - INFO     - factor_mining_workflow.factor_mining_optimization_workflow:112 - LLMs: factor=factor_generator, cod

### Running with Different Factor Types

In [20]:
# Run the factor optimization workflow via CLI
!cd .. && nat run --config_file configs/config-optimization.yml --input "volatility factors"

2026-03-03 14:33:49 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'configs/config-optimization.yml'
2026-03-03 14:33:51 - INFO     - phoenix.config:2716 - 📋 Ensuring phoenix working directory: /Users/phuo/.phoenix
2026-03-03 14:33:52 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.schemas
2026-03-03 14:33:52 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.tables
2026-03-03 14:33:52 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.types
2026-03-03 14:33:52 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.constraints
2026-03-03 14:33:52 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.defaults
2026-03-03 14:33:52 - INFO     - alembic.runtime.plugins:37 - setup plugin alembic.autogenerate.comments
2026-03-03 14:33:57 - INFO     - factor_mining_workflow.factor_mining_optimization_workflow:112 - LLMs: factor=factor_generator, cod

In [ ]:
# Run the factor optimization workflow via CLI
!cd .. && nat run --config_file configs/config-optimization.yml --input "mean-reversion factors"

<a id="interpreting-results"></a>
# 5) Interpreting Results

<a id="evaluation-metrics"></a>
## 5.1) Understanding Evaluation Metrics

<div style="background-color: #e3f2fd; border-left: 6px solid #1976d2; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📊 Backtesting Metrics</h4>
The Eval Agent evaluates factors using several key metrics:
<table style="width: 100%; border-collapse: collapse; margin-top: 10px;">
<tr style="background-color: #bbdefb;"><th style="padding: 8px; text-align: left;">Metric</th><th style="padding: 8px; text-align: left;">Description</th><th style="padding: 8px; text-align: left;">Good Value</th></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><strong>Mean IC</strong></td><td style="padding: 8px;">Average Spearman correlation between factor and forward returns</td><td style="padding: 8px;">|IC| > 0.03</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><strong>IC Std</strong></td><td style="padding: 8px;">Standard deviation of IC values</td><td style="padding: 8px;">Lower is more consistent</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><strong>IC IR</strong></td><td style="padding: 8px;">Information Ratio = Mean IC / IC Std</td><td style="padding: 8px;">> 0.5 is good</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><strong>T-statistic</strong></td><td style="padding: 8px;">Statistical significance of mean IC</td><td style="padding: 8px;">|t| > 2 is significant</td></tr>
<tr style="border-bottom: 1px solid #ddd;"><td style="padding: 8px;"><strong>P-value</strong></td><td style="padding: 8px;">Probability IC is different from zero</td><td style="padding: 8px;">< 0.05 is significant</td></tr>
<tr><td style="padding: 8px;"><strong>Positive IC Ratio</strong></td><td style="padding: 8px;">Fraction of periods with positive IC</td><td style="padding: 8px;">> 0.55 is good</td></tr>
</table>
</div>

<div style="background-color: #f9f9f9; border: 2px solid #ddd; padding: 20px; border-radius: 8px; margin: 20px 0;">
<h4 style="margin-top: 0;">📈 Rank IC Value Interpretation</h4>
<table style="width: 100%; border-collapse: collapse; margin-top: 10px;">
<tr style="background-color: #ffcdd2;"><td style="padding: 10px; width: 20%;"><strong>< 0.02</strong></td><td style="padding: 10px;"><strong>Weak/Noise:</strong> Very little predictive power. May still be useful in a multi-factor model, but not a strong standalone signal.</td></tr>
<tr style="background-color: #c8e6c9;"><td style="padding: 10px;"><strong>0.02 – 0.05</strong></td><td style="padding: 10px;"><strong>Good/Standard:</strong> Target range for institutional-grade factors (Value, Quality). Generates consistent Alpha across a large stock universe.</td></tr>
<tr style="background-color: #a5d6a7;"><td style="padding: 10px;"><strong>0.05 – 0.15</strong></td><td style="padding: 10px;"><strong>Strong:</strong> Very high-quality signal, typical of short-term Momentum or specialized High-Frequency factors.</td></tr>
<tr style="background-color: #fff3cd;"><td style="padding: 10px;"><strong>> 0.15</strong></td><td style="padding: 10px;"><strong>Exceptional/Suspicious:</strong> Rare in live markets. Check for look-ahead bias or overfitting.</td></tr>
</table>
</div>

In [8]:
import json

def interpret_ic_results(result_json: str):
    """
    Interpret the IC evaluation results.
    """
    result = json.loads(result_json)
    
    print("=" * 60)
    print("FACTOR EVALUATION RESULTS")
    print("=" * 60)
    
    status = result.get("status", "unknown")
    print(f"\nStatus: {status.upper()}")
    print(f"Iterations: {result.get('iterations', 'N/A')}")
    
    metrics = result.get("evaluation_metrics", {})
    if metrics:
        print("\nEvaluation Metrics:")
        print("-" * 40)
        print(f"  Mean IC: {metrics.get('mean_ic', 'N/A'):.4f}" if metrics.get('mean_ic') else "  Mean IC: N/A")
        print(f"  IC Std: {metrics.get('ic_std', 'N/A'):.4f}" if metrics.get('ic_std') else "  IC Std: N/A")
        print(f"  IC IR: {metrics.get('ic_ir', 'N/A'):.4f}" if metrics.get('ic_ir') else "  IC IR: N/A")
        print(f"  P-value: {metrics.get('p_value', 'N/A'):.4f}" if metrics.get('p_value') else "  P-value: N/A")
        print(f"  Positive IC Ratio: {metrics.get('positive_ic_ratio', 'N/A'):.2%}" if metrics.get('positive_ic_ratio') else "  Positive IC Ratio: N/A")
    
    if result.get("saved_path"):
        print(f"\nResults saved to: {result['saved_path']}")
    
    print("\n" + result.get("message", ""))

# Example usage (uncomment after running the workflow)
# interpret_ic_results(result)

<a id="analyzing-factors"></a>
## 5.2) Analyzing Generated Factors

Let's look at examples of saved factor results:

In [ ]:
import json
from pathlib import Path
import os

# List saved factors
output_dir = Path("../src/factor_mining_workflow/output")
if output_dir.exists():
    factor_files = list(output_dir.glob("factor_*.json"))
    print(f"Found {len(factor_files)} saved factors:\n")
    
    for f in factor_files:  # Show first 3
        print(f"\n{'='*60}")
        print(f"File: {f.name}")
        print("="*60)
        
        with open(f, "r") as file:
            data = json.load(file)
            print(f"Timestamp: {data.get('timestamp', 'N/A')}")
            print(f"Iteration: {data.get('iteration', 'N/A')}")
            
            metrics = data.get('evaluation_metrics', {})
            if metrics:
                print(f"Mean IC: {metrics.get('mean_ic', 'N/A')}")
                print(f"P-value: {metrics.get('p_value', 'N/A')}")
else:
    print("No saved factors found. Run the workflow first!")

Found 9 saved factors:


File: factor_20260203_224353_iter3.json
Timestamp: 20260203_224353
Iteration: 3
Mean IC: 0.00514029262593512
P-value: 0.15077686173154903

File: factor_20260203_225357_iter1.json
Timestamp: 20260203_225357
Iteration: 1
Mean IC: -0.008101851407789767
P-value: 0.0008689790185862911

File: factor_20260203_235636_iter1.json
Timestamp: 20260203_235636
Iteration: 1
Mean IC: -0.003063733161265774
P-value: 0.07921501040867884

File: factor_20260204_004125_iter3.json
Timestamp: 20260204_004125
Iteration: 3
Mean IC: -0.009346960509742496
P-value: 0.00016183403826408593

File: factor_20260204_004407_iter3.json
Timestamp: 20260204_004407
Iteration: 3
Mean IC: -0.0040132838831864835
P-value: 0.07718153092827462

File: factor_20260204_004911_iter1.json
Timestamp: 20260204_004911
Iteration: 1
Mean IC: -0.013710019974056726
P-value: 2.418123799197147e-06

File: factor_20260204_005642_iter3.json
Timestamp: 20260204_005642
Iteration: 3
Mean IC: -0.005392609789735757
P-value: 0.0

<a id="next-steps"></a>
# 6) Next Steps

<div style="background-color: #e8f5e9; border-left: 6px solid #4caf50; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🔮 Ideas for Extending the Workflow</h4>
<ol>
<li><strong>Add more operators</strong> — Extend <code>calculator.json</code> with additional technical indicators</li>
<li><strong>Use different data</strong> — Replace SP500 data with your own stock universe</li>
<li><strong>Customize evaluation</strong> — Modify IC thresholds or add additional metrics like turnover</li>
<li><strong>Add more data fields</strong> — Include fundamental data like earnings, book value, etc.</li>
<li><strong>Build factor portfolios</strong> — Use accepted factors to construct trading strategies</li>
</ol>
</div>

<div style="background-color: #e3f2fd; border-left: 6px solid #1976d2; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📚 Additional Resources</h4>
<ul>
<li><a href="https://docs.nvidia.com/nemo-agent-toolkit/">NeMo Agent Toolkit Documentation</a></li>
<li><a href="https://arize.com/docs/phoenix">Arize Phoenix Documentation</a></li>
</ul>
</div>